# Smart Buildings — Submission v2: feedback HVAC policy

**Competition notebook:** outputs a control policy through the official Kaggle inference server (not a CSV/Parquet prediction file). Runs on CPU with internet disabled. There are no external model files, package installs, Kaggle secrets, or references to hidden building/weather identifiers.

Version 1 always emitted zeros for *all* actions, giving a recorded public score of **1.40815** (lower is better). This version implements an adaptive, per-building HVAC controller.

Its design follows the physical plant and the organizer's PID/economizer/hot-water reset reference architecture: read sensor-derived comfort limits and denormalize zone temperatures; map zone actuators to their owning AHU; idle satisfied AHUs; modulate AHU supply temperature and static pressure by proportional/integral/derivative error; use a cooling economizer when outdoor air is favorable; control each VAV damper/reheat valve independently; and reset hot-water temperature according to real heat demand.

This is a **new, independent implementation**. Local synthetic action-interface tests passed; a full 8,640-step simulator benchmark and an official Kaggle score have **not** yet been measured, so outperforming v1 on hidden scenarios remains to be verified. Avoid confusing the organizer's baseline scores with measurements for this notebook.

Reference: https://github.com/smart-buildings-2026/starter-kit ; https://www.kaggle.com/competitions/neurips-2026-smart-buildings


In [ ]:
"""Self-contained transfer controller for the NeurIPS 2026 Smart Buildings gateway."""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Per-zone feedback, per-AHU PID, economizer and hot-water reset.

    Unlike Version 1, the controller conditions zones when they need heating
    or cooling, but otherwise idles plant equipment. All indices and sizes
    are read from the *current building's* public runtime metadata.
    """

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.n_cont = int(m["n_continuous_action"])
        self.n_disc = int(m["n_discrete_action"])
        self.fields = {name: i for i, name in enumerate(m["observation_field_names"])}
        self.temp_indices = list(map(int, m["temp_indices"]))
        self.pressure_indices = list(map(int, m["pressure_indices"]))
        self.recirc_indices = list(map(int, m["recirculation_indices"]))
        self.water_indices = list(map(int, m["water_indices"]))
        self.n_ahu = len(self.temp_indices)
        self.water_pressure_idx = (self.pressure_indices[self.n_ahu]
                                   if len(self.pressure_indices) > self.n_ahu else None)
        self.water_temp_idx = self.water_indices[0] if self.water_indices else None
        self.water_disc_idx = self.n_disc - 1
        self.comfort_band = tuple(m["comfort_temp_window"])
        self.eco_band = tuple(m["eco_temp_window"])
        self.morning = float(m["morning_start_hour"])
        self.evening = float(m["evening_start_hour"])
        stats = m.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(stats.get("sample_mean", 0.0))
        self.zone_std = float(stats.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        index_map = m["zone_action_indices"]
        assignments = m["ahu_zone_assignments"]
        def numeric_key(name):
            match = re.search(r"\d+", str(name))
            return (int(match.group()) if match else 0, str(name))
        ahu_ids = sorted(assignments, key=numeric_key)
        self.ahus = []
        for i, ahu in enumerate(ahu_ids[:self.n_ahu]):
            zones = []
            for room in assignments[ahu]:
                vav = str(room).replace("room_", "VAV_")
                obs_idx = self.fields.get(f"{vav}_zone_air_temperature_sensor")
                if obs_idx is None:
                    continue
                indices = index_map.get(vav, {})
                zones.append((
                    obs_idx,
                    self.fields.get(f"{vav}_zone_air_heating_temperature_setpoint_sensor"),
                    self.fields.get(f"{vav}_zone_air_cooling_temperature_setpoint_sensor"),
                    indices.get("supply_air_damper_percentage_command"),
                    indices.get("reheat_valve_percentage_command"),
                ))
            self.ahus.append({
                "zones": zones,
                "temp": self.temp_indices[i],
                "pressure": self.pressure_indices[i] if i < len(self.pressure_indices) else None,
                "recirc": self.recirc_indices[i] if i < len(self.recirc_indices) else None,
                "outside": self.fields.get(f"ahs_{ahu}_outside_air_temperature_sensor"),
                "run": i,
            })
        self.reset()

    def reset(self):
        self.integral = np.zeros(len(self.ahus), dtype=np.float64)
        self.previous = np.zeros(len(self.ahus), dtype=np.float64)

    def _schedule_band(self, obs):
        def phase(s, c, period):
            if s not in self.fields or c not in self.fields:
                return None
            theta = np.arctan2(float(obs[self.fields[s]]), float(obs[self.fields[c]]))
            return (theta % (2 * np.pi)) * period / (2 * np.pi)
        hour = phase("hod_sin_000", "hod_cos_000", 24)
        day = phase("dow_sin_000", "dow_cos_000", 7)
        is_work = (hour is not None and day is not None and day < 5
                   and self.morning <= hour < self.evening)
        return self.comfort_band if is_work else self.eco_band

    def _zones(self, obs, ahu, fallback):
        result = []
        for ti, li, hi, damper_idx, reheat_idx in ahu["zones"]:
            t = float(obs[ti]) * self.zone_std + self.zone_mean
            low = float(obs[li]) if li is not None else fallback[0]
            high = float(obs[hi]) if hi is not None else fallback[1]
            if not (np.isfinite(low) and np.isfinite(high) and high > low):
                low, high = fallback
            if not np.isfinite(t):
                continue
            error = low - t if t < low else (high - t if t > high else 0.0)
            result.append((t, low, high, error, damper_idx, reheat_idx))
        return result

    @staticmethod
    def _clip(x, lo=-1.0, hi=1.0):
        return float(np.clip(x, lo, hi))

    def get_action(self, obs):
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        fallback_band = self._schedule_band(obs)
        continuous = np.full(self.n_cont, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_disc, dtype=np.int32)
        heating_shortfalls = []
        heating_efforts = []
        active = False
        heating = False

        # A normalized 0 is HALF-OPEN; explicitly initialize resting commands
        # to -1 (lowest physical setpoint), not to zero.
        for i, ahu in enumerate(self.ahus):
            zone_data = self._zones(obs, ahu, fallback_band)
            if not zone_data:
                self.integral[i] *= 0.5
                self.previous[i] = 0.0
                continue
            for t, low, high, error, damper_idx, valve_idx in zone_data:
                heating_shortfalls.append(max(0.0, low - t))
                if damper_idx is not None:
                    continuous[damper_idx] = 2.0 * min(1.0, abs(error) / 2.0) - 1.0
                if valve_idx is not None:
                    continuous[valve_idx] = 2.0 * min(1.0, max(0.0, error) / 2.0) - 1.0

            mean_temp = float(np.mean([z[0] for z in zone_data]))
            mean_low = float(np.mean([z[1] for z in zone_data]))
            mean_high = float(np.mean([z[2] for z in zone_data]))
            error = (mean_low - mean_temp if mean_temp < mean_low else
                     mean_high - mean_temp if mean_temp > mean_high else 0.0)
            if error == 0.0:
                self.integral[i] *= 0.5
                self.previous[i] = 0.0
                if ahu["recirc"] is not None:
                    continuous[ahu["recirc"]] = 1.0
                continue
            self.integral[i] = np.clip(self.integral[i] + error, -10.0, 10.0)
            derivative = error - self.previous[i]
            self.previous[i] = error
            u = self._clip(0.5 * error + 0.02 * self.integral[i] + 0.1 * derivative)
            continuous[ahu["temp"]] = u
            pressure = self._clip(2.0 * abs(u) - 1.0)
            if ahu["pressure"] is not None:
                continuous[ahu["pressure"]] = pressure
            discrete[ahu["run"]] = 1
            active = True
            if error > 0:
                heating = True
            heating_efforts.append(max(0.0, u))

            if ahu["recirc"] is not None:
                recirc_cmd = 1.0
                oi = ahu["outside"]
                if u < 0.0 and oi is not None:
                    # Outside and zone setpoint sensors are in raw kelvin;
                    # zone air temperature is normalized and denormalized above.
                    outside = float(obs[oi])
                    inside = mean_temp
                    pressure_pa = max(200.0, 1000.0 * (pressure + 1.0))
                    if np.isfinite(outside) and inside - outside > pressure_pa * 0.00322:
                        target = max(mean_low, 285.0 + 10.0 * (u + 1.0))
                        if inside > target:
                            outdoor_fraction = self._clip(
                                (inside - target) / max(1e-6, inside - outside), 0.0, 1.0)
                            recirc_cmd = 1.0 - 2.0 * outdoor_fraction
                continuous[ahu["recirc"]] = recirc_cmd

        if self.water_temp_idx is not None and heating:
            demand = (float(np.mean(heating_shortfalls)) / 4.0
                      if heating_shortfalls else 0.0)
            saturation = (float(np.quantile(heating_efforts, 0.25))
                          if heating_efforts else 0.0)
            continuous[self.water_temp_idx] = 2.0 * self._clip(
                max(demand, saturation), 0.0, 1.0) - 1.0
        if self.water_pressure_idx is not None:
            continuous[self.water_pressure_idx] = 1.0 if active else -1.0
        if 0 <= self.water_disc_idx < self.n_disc:
            discrete[self.water_disc_idx] = int(active)

        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# Kaggle's gateway performs the public smoke rollout when run interactively,
# and the hidden 30-scenario evaluation when submitted.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
